In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('exoplanet-detection-challenge')

print("Path to competition files:", path)

/Users/rishabhyadav/techrelatedfolder/ai&mlShit/exoplanet_detection_challenge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: /Users/rishabhyadav/.cache/kagglehub/competitions/exoplanet-detection-challenge


In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


train= pd.read_csv(path + '/train.csv')
test= pd.read_csv(path + '/test.csv')
sub= pd.read_csv(path + '/sample_submission.csv')

train.set_index('star_id', inplace=True)
test.set_index('star_id', inplace=True)

train.head()

,spectral_type,stellar_radius_sr,stellar_mass_sm,stellar_teff_k,stellar_log_g,stellar_luminosity,stellar_metallicity,stellar_rot_period_d,stellar_noise_ppm,label,...,impact_parameter,transit_depth_ppm,transit_duration_hr,n_transits_observed,orbital_velocity_kms,transit_snr,planet_eq_temp_k,flux_variability_index,log_period,log_snr
star_id,,,,,,,,,,,,,,,,,,,,,
STR-000436,F,1.327806,1.398391,6912.100561,4.323038,3.612614,-0.178,21.07,250.66,0,...,2.183038,22.901432,0.155303,0.0,NaN,0.730875,NaN,0.0914,0.0000,0.5486
STR-007250,K,0.912461,0.708236,4533.876776,4.352924,0.298579,-0.067,16.43,236.22,0,...,1.326354,2510.133991,13.584701,0.0,NaN,4.513105,NaN,10.6263,3.5789,1.7071
STR-005589,K,0.684165,0.716614,4466.620216,4.652811,0.146280,0.083,39.55,162.24,1,...,0.539900,5076.490629,2.646656,56.0,79.732,235.808062,503.8,31.2900,2.6701,5.4672
STR-011770,B,4.403350,9.499783,22146.527056,4.042372,4379.887860,-0.056,15.55,615.27,0,...,1.326998,20.652450,0.022175,0.0,NaN,1.261956,NaN,0.0336,0.0000,0.8162
STR-003949,G,0.927414,0.974574,5647.596422,4.606604,0.753481,0.194,9.44,268.57,1,...,0.479000,334.291352,NaN,46.0,110.276,7.598188,839.8,1.2447,2.1764,2.1516


In [ ]:
X=train.drop(columns=['label'])
y=train['label']
X_test=test.copy()

for df in [X,X_test]:
    for col in ['planet_radius_re','orbital_period_d','planet_eq_temp_k','eccentricity','semi_major_au','rp_rs_ratio']:
        if col in df.columns:
            df[col+'_missing']=df[col].isna().astype(int)
    if set(['rp_rs_ratio','stellar_radius_sr']).issubset(df.columns):
        m=df['planet_radius_re'].isna()
        df.loc[m,'planet_radius_re']=df.loc[m,'rp_rs_ratio']*df.loc[m,'stellar_radius_sr']*109.2
    if 'rp_rs_ratio' in df.columns:
        df['expected_depth']=df['rp_rs_ratio']**2
    if set(['transit_snr','stellar_noise_ppm']).issubset(df.columns):
        df['snr_noise']=df['transit_snr']/(df['stellar_noise_ppm']+1)
    if set(['transit_depth_ppm','stellar_noise_ppm']).issubset(df.columns):
        df['noise_ratio']=df['transit_depth_ppm']/(df['stellar_noise_ppm']+1)
    if set(['transit_duration_hr','orbital_period_d']).issubset(df.columns):
        df['duration_per_period']=df['transit_duration_hr']/(df['orbital_period_d']+1e-6)


In [ ]:
cat=[c for c in X.columns if X[c].dtype=='str']
num=[c for c in X.columns if c not in cat]

pre=ColumnTransformer([
('num',SimpleImputer(strategy='median'),num),
('cat',Pipeline([
('imp',SimpleImputer(strategy='most_frequent')),
('oh',OneHotEncoder(handle_unknown='ignore'))
]),cat)
])

models = {
    'Random_Forest': RandomForestClassifier(random_state = 67, class_weight = 'balanced'),
    'Logistic_Regression' : LogisticRegression(max_iter = 1000, random_state = 67, class_weight = 'balanced'),
    'SVC' : SVC(probability = True, random_state = 67, class_weight = 'balanced'),
    'CatBoost_Classifier' : CatBoostClassifier(iterations = 1000, random_state = 67, learning_rate = 0.05, depth = 6, verbose = False)
    }

metrics_evaluation_result = [] 
best_model = None
best_model_name = None
best_mean_score = 0

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro',
    'roc_auc': 'roc_auc_ovo' # use 'roc_auc' for binary classification
}

for model_name, model in models.items():

    pipe = Pipeline([('pre',pre),('model',model)])

    cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 67)
    scores = cross_validate(pipe, X, y, cv = cv, scoring = scoring_metrics)
    metrics_evaluation_result.append({
        'Model': model_name,
        'Mean Accuracy Score' : np.mean(scores['test_accuracy']),
        'Precision' : np.mean(scores['test_precision']),
        'Recall' : np.mean(scores['test_recall']),
        'F1': np.mean(scores['test_f1']),
        'ROC_AUC' : np.mean(scores['test_roc_auc'])
    })

    if best_model == None:
        best_model = pipe
        best_mean_score = np.mean(scores['test_accuracy'])
    else:
        if np.mean(scores['test_accuracy']) >= best_mean_score:
            best_model = pipe
            best_mean_score = np.mean(scores['test_accuracy'])
            best_model_name = model_name

metrics_evaluation = pd.DataFrame(metrics_evaluation_result)
metrics_evaluation.set_index('Model', inplace=True)
metrics_evaluation

{'fit_time': array([0.44377613, 0.44788027, 0.45772815, 0.42998314, 0.43730307]), 'score_time': array([0.02987313, 0.03079605, 0.03053212, 0.02827072, 0.02933407]), 'test_accuracy': array([1., 1., 1., 1., 1.]), 'test_precision': array([1., 1., 1., 1., 1.]), 'test_recall': array([1., 1., 1., 1., 1.]), 'test_f1': array([1., 1., 1., 1., 1.]), 'test_roc_auc': array([1., 1., 1., 1., 1.])}
{'fit_time': array([0.09329987, 0.08711195, 0.11490011, 0.08944941, 0.0848341 ]), 'score_time': array([0.01434016, 0.01565838, 0.01592684, 0.01499796, 0.014678  ]), 'test_accuracy': array([1.        , 0.99944444, 1.        , 1.        , 1.        ]), 'test_precision': array([1.        , 0.99963689, 1.        , 1.        , 1.        ]), 'test_recall': array([1.        , 0.99882075, 1.        , 1.        , 1.        ]), 'test_f1': array([1.        , 0.99922806, 1.        , 1.        , 1.        ]), 'test_roc_auc': array([1.        , 0.99769807, 1.        , 1.        , 1.        ])}


/Users/rishabhyadav/techrelatedfolder/ai&mlShit/exoplanet_detection_challenge/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rishabhyadav/techrelatedfolder/ai&mlShit/exoplanet_detection_challenge/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rishabhyadav/techrelatedfolder/ai&mlShit/exoplanet_detection_challenge/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(p

{'fit_time': array([3.59029913, 3.64533877, 3.37353301, 3.278795  , 3.54921079]), 'score_time': array([0.57785606, 0.5843358 , 0.55558896, 0.5290978 , 0.58054924]), 'test_accuracy': array([0.92111111, 0.93444444, 0.92166667, 0.93111111, 0.94055556]), 'test_precision': array([0.89717847, 0.9140334 , 0.90084561, 0.90999602, 0.92258283]), 'test_recall': array([0.87986781, 0.90164272, 0.87719786, 0.89638503, 0.91069519]), 'test_f1': array([0.88806735, 0.90761602, 0.88818275, 0.90291853, 0.91643929]), 'test_roc_auc': array([0.96380505, 0.97190122, 0.96832257, 0.97434866, 0.97240984])}
{'fit_time': array([2.56961393, 2.53443885, 2.63952518, 2.53269005, 2.61512327]), 'score_time': array([0.02243686, 0.02165127, 0.02137399, 0.02106285, 0.02092195]), 'test_accuracy': array([1., 1., 1., 1., 1.]), 'test_precision': array([1., 1., 1., 1., 1.]), 'test_recall': array([1., 1., 1., 1., 1.]), 'test_f1': array([1., 1., 1., 1., 1.]), 'test_roc_auc': array([1., 1., 1., 1., 1.])}


,Mean Accuracy Score,Precision,Recall,F1,ROC_AUC
Model,,,,,
Random_Forest,1.000000,1.000000,1.000000,1.000000,1.000000
Logistic_Regression,0.999889,0.999927,0.999764,0.999846,0.999540
SVC,0.929778,0.908927,0.893158,0.900645,0.970157
CatBoost_Classifier,1.000000,1.000000,1.000000,1.000000,1.000000


| Model | Mean Accuracy Score | Precision | Recall | F1 | ROC_AUC |
|------------------------|--------------------:|----------:|-------:|-------:|---------:|
| Random_Forest | 1.000000 | 1.000000 | 1.000000 | 1.000000 | 1.000000 |
| Logistic_Regression | 0.999889 | 0.999927 | 0.999764 | 0.999846 | 0.999540 |
| SVC | 0.929778 | 0.908927 | 0.893158 | 0.900645 | 0.970157 |
| CatBoost_Classifier | 1.000000 | 1.000000 | 1.000000 | 1.000000 | 1.000000 |

In [6]:
best_model.fit(X, y)

pred = best_model.predict(X_test)
sub['label']=pred.astype(int)
sub.to_csv('submission.csv',index=False)
print('Preview of Predicted Labels for Exoplanet Detection')
print(sub.head(10))
print('Prediction Distribution: ')
print(sub['label'].value_counts())
print(f'Best model was found to be {best_model_name}')

Preview of Predicted Labels for Exoplanet Detection
      star_id  label
0  STR-011368      0
1  STR-005379      0
2  STR-004743      1
3  STR-007357      0
4  STR-010554      0
5  STR-002573      1
6  STR-004111      0
7  STR-002827      0
8  STR-009245      0
9  STR-007693      0
Prediction Distribution: 
label
0    2293
1     707
Name: count, dtype: int64
Best model was found to be CatBoost_Classifier
